In [15]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

In [16]:
np.random.seed(42)
tf.random.set_seed(42)

# Chuẩn bị dữ liệu


Ta sử dụng dataset **Emotions dataset for NLP** ([Nguồn](https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp)).
Dữ liệu đã được xử lý trước để thuận tiện cho việc xây mô hình.


In [17]:
df = pd.read_csv("../data/NLP.csv")
df.head()

,statement,emotion
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


Để xây mô hình, ta cần gán các cảm xúc cần dự đoán cho một chỉ số `emotion id` như dưới đây:


In [18]:
emotion_map = {
    "surprise": 0,
    "joy": 1,
    "sadness": 2,
    "anger": 3,
    "fear": 4,
    "love": 5,
}

df['emotion id'] = df['emotion'].map(emotion_map)
df.head()

,statement,emotion,emotion id
0,im feeling rather rotten so im not very ambiti...,sadness,2
1,im updating my blog because i feel shitty,sadness,2
2,i never make her separate from me because i do...,sadness,2
3,i left with my bouquet of red and yellow tulip...,joy,1
4,i was feeling a little vain when i did this one,sadness,2


Ta xáo trộn dữ liệu trước khi tách để giúp tách tập dữ liệu cân đối hơn.


In [19]:
df = df.sample(frac=1).reset_index(drop=True)
df.head(10)

,statement,emotion,emotion id
0,i feel assured that foods that are grown organ...,joy,1
1,i already have my christmas trees up i got two...,joy,1
2,i feel all betrayed and disillusioned,sadness,2
3,i will tell you that i am feeling quite invigo...,joy,1
4,i start to feel less exhausted the bits and pi...,sadness,2
5,i was listening to belle and sebastian feeling...,fear,4
6,i be able to look them in the face again witho...,sadness,2
7,i am thankful for feeling useful,joy,1
8,i woke up feeling artistic ish,joy,1
9,i was taunted by the ability of feeling threat...,fear,4


Tiến hành chia tập dữ liệu:

- Dữ liệu chia thành hai tập `raw` và `test` với tỉ lệ 8:2,
- Sau đó, chia tập `raw` thành `train` và `val` với tỉ lệ 7:1.

Như vậy, tỉ lệ chia dữ liệu là `train`:`val`:`test` = 7:1:2.


In [20]:
from sklearn.model_selection import train_test_split

X = df['statement'].values
y = df['emotion id'].values

X_raw, X_test, y_raw, y_test = train_test_split(X,y,
                                                   test_size = 0.2,
                                                   random_state = 42)
X_raw = np.array(X_raw)
X_test = np.array(X_test)
y_raw = np.array(y_raw)
y_test = np.array(y_test)

X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw,
                                                 test_size=0.125,
                                                 random_state=42)

X_train = np.array(X_train)
X_val = np.array(X_val)
y_train = np.array(y_train)
y_val = np.array(y_val)

X_train.shape, X_val.shape, X_test.shape

((14000,), (2000,), (4000,))

# Xây dựng mô hình


Ta sử dụng tokenizer trước để xử lý các từ/cụm từ/kí tự đặc biệt thành số mà máy tính có thể hiểu được. Đây là quá trình **tokenize**


In [21]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

vocab_size = 10000
max_length = 150

tokenizer = Tokenizer(num_words = vocab_size, oov_token = "OOV") #out of vocabulary
tokenizer.fit_on_texts(X_train)

X_train_seqs = tokenizer.texts_to_sequences(X_train)
X_train_padded = pad_sequences(X_train_seqs, maxlen=max_length, padding="post")

X_val_seqs = tokenizer.texts_to_sequences(X_val)
X_val_padded = pad_sequences(X_val_seqs, maxlen=max_length, padding="post")

len(X_train_padded),len(X_val_padded)

(14000, 2000)

Bây giờ, ta sẽ tiến hành xây mô hình NLP.


In [22]:
num_classes = len(emotion_map)

embedding_dim = 64

model = keras.Sequential([
    keras.Input(shape=(max_length,)),
    keras.layers.Embedding(vocab_size, embedding_dim, trainable=True),
    keras.layers.Flatten(),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(optimizer='adam',
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy']
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 150, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 9600)           │        38,400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       614,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,293,254 (4.93 MB)

 Trainable params: 1,274,054 (4.86 MB)

 Non-trainable params: 19,200 (75.00 KB)

# Huấn luyện mô hình


Ta bắt đầu huấn luyện mô hình với dữ liệu train và val đã xử lý ở trên.


In [24]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath = "NLP_ep{epoch:02d}.h5",
    save_weights_only = False,
    save_best_only = False,
    monitor = "val_loss",
    verbose = 1
)

history = model.fit(
    X_train_padded, y_train,
    validation_data = (X_val_padded, y_val),
    epochs = 15,
    callbacks = [checkpoint],
)

Epoch 1/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9638 - loss: 0.1067
Epoch 1: saving model to NLP_ep01.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9660 - loss: 0.1040 - val_accuracy: 0.8190 - val_loss: 0.7930
Epoch 2/15
435/438 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9745 - loss: 0.0806
Epoch 2: saving model to NLP_ep02.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9753 - loss: 0.0762 - val_accuracy: 0.8240 - val_loss: 0.8189
Epoch 3/15
434/438 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9770 - loss: 0.0735
Epoch 3: saving model to NLP_ep03.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9769 - loss: 0.0776 - val_accuracy: 0.8190 - val_loss: 0.9677
Epoch 4/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9772 - loss: 0.0720
Epoch 4: saving model to NLP_ep04.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9774 - loss: 0.0706 - val_accuracy: 0.8130 - val_loss: 0.9620
Epoch 5/15
434/438 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9794 - loss: 0.0684
Epoch 5: saving model to NLP_ep05.h5


...

438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9835 - loss: 0.0548 - val_accuracy: 0.8225 - val_loss: 1.1452
Epoch 12/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9833 - loss: 0.0567
Epoch 12: saving model to NLP_ep12.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9845 - loss: 0.0508 - val_accuracy: 0.8390 - val_loss: 1.1766
Epoch 13/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9852 - loss: 0.0606
Epoch 13: saving model to NLP_ep13.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9857 - loss: 0.0505 - val_accuracy: 0.8325 - val_loss: 1.2104
Epoch 14/15
436/438 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9851 - loss: 0.0432
Epoch 14: saving model to NLP_ep14.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9864 - loss: 0.0441 - val_accuracy: 0.8220 - val_loss: 1.1380
Epoch 15/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9835 - loss: 0.0634
Epoch 15: saving model to NLP_ep15.h5


438/438 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9864 - loss: 0.0479 - val_accuracy: 0.8200 - val_loss: 1.3103


# Kiểm tra mô hình


Bây giờ, ta kiểm tra độ chính xác của mô hình bằng cách kiểm tra.


In [25]:
X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_seqs, padding='post', truncating='post', maxlen=max_length)

In [26]:
from tensorflow.keras.models import load_model

model = load_model("NLP_ep15.h5")
preds = model.predict(X_test_padded)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


# Đánh giá mô hình


Ta tiến hành đánh giá mô hình trên bằng các chỉ số:

- `accuracy`: độ chính xác của mô hình.
- `f1-score`: chỉ số đánh giá mô hình bằng cách kết hợp Precision (độ chính xác) và Recall (độ bao phủ). Nó được tính bằng trung bình điều hòa của hai giá trị này:


In [27]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

pred_classes = np.argmax(preds, axis=1)

accuracy = accuracy_score(y_test, pred_classes)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

f1 = f1_score(y_test, pred_classes, average='weighted')
print(f"Weighted F1-Score: {f1 * 100:.2f}%")


Test Accuracy: 80.53%

Weighted F1-Score: 80.00%
